# 🌊 PROYECTO CENTINELA — FASE 2
## Rama C (CORREGIDA): Fusión Multimodal con Alineación Declarada

---

### ¿Qué cambió respecto a la versión anterior?

La versión anterior emparejaba la ventana temporal *i* con la imagen *i* mediante
`min(len(gru), len(cnn))`. Ese **pareo artificial por muestra** es una forma de
**fuga de datos (leakage)** señalada explícitamente en la actividad, y además
truncaba el conjunto de fusión a ~78 muestras de prueba, lo que hizo colapsar el
modelo (ROC-AUC 0.31, Recall 0.0). Esta versión lo corrige de raíz.

### 🔑 Declaración de alineación (la decisión más importante de la Rama C)

**No existe una clave genuina por muestra** entre las dos fuentes: la serie del UCI
(*Water Quality Prediction*, 37 estaciones × 705 días) no trae imágenes, y el
dataset visual (Roboflow, agua Limpia/Turbia/Contaminada) no trae fecha ni GPS.
Por lo tanto:

1. **Se declara una alineación simulada por condición**: a cada ventana temporal se
   le asigna una imagen muestreada aleatoriamente del *pool* cuya clase visual es
   coherente con el **estado observable del agua en el último día de la ventana**
   (pH dentro de rango → Limpia; pH fuera de rango → Turbia o Contaminada).
2. La clave de emparejamiento es el **estado actual observable** (día *t*), nunca la
   **etiqueta objetivo** (riesgo en *t+1*). Emparejar con la etiqueta futura sería
   fuga de datos; emparejar con el estado presente simula lo que un operario del
   acueducto fotografiaría hoy.
3. Las imágenes de entrenamiento y prueba provienen de *pools* disjuntos (la misma
   partición 70/15/15 de la Rama A), de modo que ninguna imagen de prueba fue vista
   en entrenamiento.

**Limitación declarada:** la correlación imagen–serie es simulada, no medida en
campo. El resultado de la fusión demuestra la *arquitectura* multimodal; su valor
real exigiría co-registro genuino (finca/bocatoma, fecha), que se propone como
trabajo de la Fase 3.


---
## BLOQUE 1 — Librerías y configuración
### ¿Qué hace?
Importa librerías, fija la semilla y define las rutas. En Colab, sube
`water_dataset.mat`, la carpeta `Imagenes_agua/train` y los pesos
`mejor_GRU.pth` y `mejor_clf_ResNet50.pth` (o monta Drive).

In [ ]:
# ============================================================
# BLOQUE 1 — LIBRERÍAS Y CONFIGURACIÓN
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io as sio

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import datasets, transforms, models
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, recall_score, precision_score,
                             f1_score, accuracy_score, confusion_matrix)

SEMILLA = 42
torch.manual_seed(SEMILLA)
np.random.seed(SEMILLA)
DISPOSITIVO = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

RUTA_MAT      = 'water_dataset.mat'
RUTA_IMAGENES = 'Imagenes_agua/train'
RUTA_PTH_GRU  = 'mejor_GRU.pth'
RUTA_PTH_CNN  = 'mejor_clf_ResNet50.pth'

print(f'Dispositivo: {DISPOSITIVO}')

---
## BLOQUE 2 — Datos de la Rama B (serie temporal)
### ¿Qué hace?
Reconstruye las ventanas de 14 días **igual que en la Rama B**, pero ahora también
guarda la **condición observable** del último día de cada ventana (pH dentro o
fuera de rango en el día *t*). Esa condición es la clave de la alineación simulada
y es observable en el momento de la predicción — no usa la etiqueta futura.

In [ ]:
# ============================================================
# BLOQUE 2 — DATOS RAMA B + CONDICIÓN OBSERVABLE
# ============================================================
VENTANA     = 14
INDICADORES = ['SC_max', 'pH_max', 'pH_min', 'SC_min', 'SC_mean',
               'DO_max', 'DO_mean', 'DO_min', 'T_mean', 'T_min', 'T_max']
PH_MIN, PH_MAX = 6.5, 8.5
IDX_PH_MAX, IDX_PH_MIN = INDICADORES.index('pH_max'), INDICADORES.index('pH_min')

mat  = sio.loadmat(RUTA_MAT)
X_tr, Y_tr, X_te, Y_te = mat['X_tr'], mat['Y_tr'], mat['X_te'], mat['Y_te']

def construir_dataset(X_mat, Y_mat):
    filas = []
    for t in range(X_mat.shape[1] - 1):
        Xt, y_next = X_mat[0, t], Y_mat[:, t + 1]
        for s in range(Xt.shape[0]):
            ph_sig = float(y_next[s]) * 10
            riesgo = int(ph_sig < PH_MIN or ph_sig > PH_MAX)
            filas.append(Xt[s].tolist() + [riesgo, s, t])
    return pd.DataFrame(filas, columns=INDICADORES + ['riesgo', 'estacion', 'dia'])

df_train, df_test = construir_dataset(X_tr, Y_tr), construir_dataset(X_te, Y_te)

def crear_ventanas_con_condicion(df, ventana=14):
    """Devuelve X, y y la condición observable del último día de la ventana.
    cond = 1 si el pH del día t (valores normalizados ÷10) está fuera de rango."""
    X_list, y_list, cond_list = [], [], []
    for est in df['estacion'].unique():
        df_est  = df[df['estacion'] == est].sort_values('dia')
        valores = df_est[INDICADORES].values
        etiq    = df_est['riesgo'].values
        for i in range(ventana, len(valores)):
            X_list.append(valores[i-ventana:i])
            y_list.append(etiq[i])
            ph_hoy_max = valores[i-1, IDX_PH_MAX] * 10
            ph_hoy_min = valores[i-1, IDX_PH_MIN] * 10
            cond_list.append(int(ph_hoy_max > PH_MAX or ph_hoy_min < PH_MIN))
    return np.array(X_list), np.array(y_list), np.array(cond_list)

X_train_b, y_train_b, cond_train = crear_ventanas_con_condicion(df_train, VENTANA)
X_test_b,  y_test_b,  cond_test  = crear_ventanas_con_condicion(df_test,  VENTANA)

n_feats_b = X_train_b.shape[2]
scaler = StandardScaler()
X_train_b_sc = scaler.fit_transform(X_train_b.reshape(-1, n_feats_b)).reshape(X_train_b.shape)
X_test_b_sc  = scaler.transform(X_test_b.reshape(-1, n_feats_b)).reshape(X_test_b.shape)

print(f'Train: {X_train_b_sc.shape} | Test: {X_test_b_sc.shape}')
print(f'Riesgo en test: {y_test_b.mean()*100:.1f}%')
print(f'Condición "fuera de rango hoy" — train: {cond_train.mean()*100:.1f}% | test: {cond_test.mean()*100:.1f}%')
print(f'Correlación condición(t) vs riesgo(t+1) en train: {np.corrcoef(cond_train, y_train_b)[0,1]:.3f}')

---
## BLOQUE 3 — Datos de la Rama A (imágenes) y extracción de features
### ¿Qué hace?
Carga las 512 imágenes con la **misma partición 70/15/15 y semilla de la Rama A** y
extrae una sola vez los *embeddings* de 2048 dimensiones del backbone ResNet50
congelado. Luego arma los *pools* de imágenes por clase y por partición: las
ventanas de entrenamiento solo podrán recibir imágenes del *pool* de
entrenamiento y las de prueba solo del *pool* de prueba.

In [ ]:
# ============================================================
# BLOQUE 3 — FEATURES DE IMAGEN Y POOLS POR CLASE/PARTICIÓN
# ============================================================
IMG_SIZE = 224
MAPA_CLASES = {'Bersih': 'Limpia', 'Keruh': 'Turbia', 'Kotor': 'Contaminada'}

transform_val = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

dataset_img = datasets.ImageFolder(root=RUTA_IMAGENES, transform=transform_val)
clases_esp  = [MAPA_CLASES.get(c, c) for c in dataset_img.classes]
N_CLASES_CNN = len(dataset_img.classes)
targets_img  = np.array(dataset_img.targets)

total = len(dataset_img)
n_train, n_val = int(total * 0.70), int(total * 0.15)
gen = torch.Generator().manual_seed(SEMILLA)
ds_tr_img, ds_vl_img, ds_te_img = random_split(
    dataset_img, [n_train, n_val, total - n_train - n_val], generator=gen)

# Backbone congelado (una sola pasada por TODAS las imágenes)
resnet_base = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
backbone_a  = nn.Sequential(*list(resnet_base.children())[:-1], nn.Flatten()).to(DISPOSITIVO)
backbone_a.eval()

loader_all = DataLoader(dataset_img, batch_size=32, shuffle=False, num_workers=0)
feats = []
with torch.no_grad():
    for imgs, _ in loader_all:
        feats.append(backbone_a(imgs.to(DISPOSITIVO)).cpu())
F_all = torch.cat(feats)                      # (512, 2048)
N_FEATS_A = F_all.shape[1]

# Pools de índices por partición y clase (0=Limpia, 1=Turbia, 2=Contaminada)
idx_tr, idx_te = np.array(ds_tr_img.indices), np.array(ds_te_img.indices)
pool = {
    'train': {c: idx_tr[targets_img[idx_tr] == c] for c in range(N_CLASES_CNN)},
    'test':  {c: idx_te[targets_img[idx_te] == c] for c in range(N_CLASES_CNN)},
}
print(f'Features extraídos: {F_all.shape}')
for split in pool:
    print(f'Pool {split}: ' + ' | '.join(f'{clases_esp[c]}={len(pool[split][c])}' for c in range(N_CLASES_CNN)))

---
## BLOQUE 4 — Cargar los modelos entrenados de las Ramas A y B
### ¿Qué hace?
Carga el GRU (Rama B) y el clasificador ResNet50 (Rama A). Del GRU se usará el
**embedding de 64 dimensiones** del último paso; de la CNN, el embedding de 2048
del backbone. Es la fusión tardía de *embeddings* que pide la actividad
(ŷ = σ(W·[z_img ‖ z_seq] + b)), no una fusión de probabilidades.

In [ ]:
# ============================================================
# BLOQUE 4 — CARGAR MODELOS ENTRENADOS
# ============================================================
class ModeloRecurrente(nn.Module):
    def __init__(self, n_features, hidden_size=64, n_capas=2, dropout=0.3):
        super().__init__()
        self.rnn = nn.GRU(input_size=n_features, hidden_size=hidden_size,
                          num_layers=n_capas, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(self.dropout(out[:, -1, :])).squeeze(1)
    def get_embedding(self, x):
        out, _ = self.rnn(x)
        return out[:, -1, :]                      # z_seq (64) sin dropout en eval

ckpt_gru = torch.load(RUTA_PTH_GRU, map_location=DISPOSITIVO)
modelo_gru = ModeloRecurrente(n_features=n_feats_b).to(DISPOSITIVO)
modelo_gru.load_state_dict(ckpt_gru['modelo_state_dict'] if 'modelo_state_dict' in ckpt_gru else ckpt_gru)
modelo_gru.eval()
DIM_GRU = 64

ckpt_cnn = torch.load(RUTA_PTH_CNN, map_location=DISPOSITIVO)
clf_cnn = nn.Sequential(nn.Dropout(0.3), nn.Linear(N_FEATS_A, N_CLASES_CNN)).to(DISPOSITIVO)
clf_cnn.load_state_dict(ckpt_cnn['clf_state_dict'] if 'clf_state_dict' in ckpt_cnn else ckpt_cnn)
clf_cnn.eval()
print('GRU y clasificador CNN cargados correctamente')

---
## BLOQUE 5 — Alineación simulada por condición observable
### ¿Qué hace?
Asigna a cada ventana temporal una imagen del *pool* correcto:
- Condición **dentro de rango hoy** → imagen aleatoria de *Limpia*.
- Condición **fuera de rango hoy** → imagen aleatoria de *Turbia* o *Contaminada*.

El muestreo usa semilla fija (reproducible) y respeta la partición: ventanas de
train reciben imágenes de train; ventanas de test, imágenes de test. La etiqueta
objetivo (riesgo mañana) **nunca** interviene en el emparejamiento.

In [ ]:
# ============================================================
# BLOQUE 5 — EMPAREJAMIENTO SIMULADO (SIN FUGA DE DATOS)
# ============================================================
rng = np.random.default_rng(SEMILLA)

def asignar_imagenes(cond, split):
    """cond: array binario (0 = agua en rango hoy, 1 = fuera de rango hoy)."""
    indices = np.empty(len(cond), dtype=int)
    p = pool[split]
    pool_riesgo = np.concatenate([p[1], p[2]])    # Turbia + Contaminada
    mask = cond == 1
    indices[~mask] = rng.choice(p[0], size=(~mask).sum(), replace=True)
    indices[mask]  = rng.choice(pool_riesgo, size=mask.sum(), replace=True)
    return indices

img_idx_train = asignar_imagenes(cond_train, 'train')
img_idx_test  = asignar_imagenes(cond_test,  'test')

Z_img_train = F_all[img_idx_train]               # (n_train, 2048)
Z_img_test  = F_all[img_idx_test]                # (n_test, 2048)

print(f'Imágenes asignadas — train: {Z_img_train.shape} | test: {Z_img_test.shape}')
print('Verificación anti-fuga: intersección de imágenes train/test =',
      len(set(img_idx_train) & set(img_idx_test)))   # debe ser 0

---
## BLOQUE 6 — Embeddings del GRU y arquitectura de fusión
### ¿Qué hace?
Calcula z_seq (64 dim) para todas las ventanas y define la cabeza densa de
fusión: [z_img (2048) ‖ z_seq (64)] → 2112 → 64 → 1 logit de riesgo.

In [ ]:
# ============================================================
# BLOQUE 6 — EMBEDDINGS Y ARQUITECTURA DE FUSIÓN
# ============================================================
def embeddings_gru(modelo, X_np):
    X_t = torch.tensor(X_np, dtype=torch.float32)
    dl  = DataLoader(TensorDataset(X_t), batch_size=256, shuffle=False)
    zs = []
    with torch.no_grad():
        for (xb,) in dl:
            zs.append(modelo.get_embedding(xb.to(DISPOSITIVO)).cpu())
    return torch.cat(zs)

Z_seq_train = embeddings_gru(modelo_gru, X_train_b_sc)   # (n, 64)
Z_seq_test  = embeddings_gru(modelo_gru, X_test_b_sc)

X_fus_train = torch.cat([Z_img_train, Z_seq_train], dim=1)   # (n, 2112)
X_fus_test  = torch.cat([Z_img_test,  Z_seq_test],  dim=1)
y_fus_train = torch.tensor(y_train_b, dtype=torch.float32)
y_fus_test  = torch.tensor(y_test_b,  dtype=torch.float32)

class CabezaFusion(nn.Module):
    """Fusión tardía: concatena z_img y z_seq y decide con una cabeza densa."""
    def __init__(self, dim_img=2048, dim_seq=64):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(dim_img + dim_seq, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1))
    def forward(self, x):
        return self.red(x).squeeze(1)

modelo_fusion = CabezaFusion(N_FEATS_A, DIM_GRU).to(DISPOSITIVO)
print(f'Dataset fusión — train: {X_fus_train.shape} | test: {X_fus_test.shape}')
print(f'Parámetros de la cabeza: {sum(p.numel() for p in modelo_fusion.parameters()):,}')

---
## BLOQUE 7 — Entrenamiento de la fusión (con validación honesta)
### ¿Qué hace?
Separa un 15% del train como validación para el *early stopping* (la versión
anterior validaba contra el test, otra fuente de sesgo optimista) y entrena con
`pos_weight` para el desbalance de clases.

In [ ]:
# ============================================================
# BLOQUE 7 — ENTRENAMIENTO DE LA FUSIÓN
# ============================================================
n_val_f = int(len(X_fus_train) * 0.15)
perm = torch.randperm(len(X_fus_train), generator=torch.Generator().manual_seed(SEMILLA))
idx_val, idx_tr = perm[:n_val_f], perm[n_val_f:]

dl_tr = DataLoader(TensorDataset(X_fus_train[idx_tr], y_fus_train[idx_tr]),
                   batch_size=128, shuffle=True)
dl_vl = DataLoader(TensorDataset(X_fus_train[idx_val], y_fus_train[idx_val]),
                   batch_size=256, shuffle=False)

pos_weight = torch.tensor([(y_fus_train[idx_tr] == 0).sum() /
                           (y_fus_train[idx_tr] == 1).sum()]).to(DISPOSITIVO)
criterio    = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizador = torch.optim.Adam(modelo_fusion.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler   = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizador, mode='min',
                                                          patience=5, factor=0.5)
EPOCAS, PACIENCIA = 60, 10
mejor_loss, sin_mejora = float('inf'), 0
hist = {'train': [], 'val': []}

for epoca in range(1, EPOCAS + 1):
    modelo_fusion.train(); lt = 0
    for xb, yb in dl_tr:
        xb, yb = xb.to(DISPOSITIVO), yb.to(DISPOSITIVO)
        optimizador.zero_grad()
        loss = criterio(modelo_fusion(xb), yb)
        loss.backward(); optimizador.step(); lt += loss.item()
    modelo_fusion.eval(); lv = 0
    with torch.no_grad():
        for xb, yb in dl_vl:
            lv += criterio(modelo_fusion(xb.to(DISPOSITIVO)), yb.to(DISPOSITIVO)).item()
    loss_t, loss_v = lt/len(dl_tr), lv/len(dl_vl)
    hist['train'].append(loss_t); hist['val'].append(loss_v)
    scheduler.step(loss_v)
    if loss_v < mejor_loss:
        mejor_loss, sin_mejora = loss_v, 0
        torch.save(modelo_fusion.state_dict(), 'mejor_fusion.pth')
    else:
        sin_mejora += 1
    if epoca % 10 == 0:
        print(f'   Época {epoca:3d} | Loss train: {loss_t:.4f} | Loss val: {loss_v:.4f}')
    if sin_mejora >= PACIENCIA:
        print(f'   Early stopping en época {epoca}'); break

modelo_fusion.load_state_dict(torch.load('mejor_fusion.pth', map_location=DISPOSITIVO))
print(f'Fusión entrenada. Mejor loss val: {mejor_loss:.4f}')

---
## BLOQUE 8 — Estudio de ablación: solo-serie / solo-imagen / fusión
### ¿Qué hace?
Compara las tres configuraciones **sobre exactamente el mismo conjunto de prueba**
(las 3.020 ventanas, no 78 como en la versión anterior):
- **Solo serie:** el GRU original de la Rama B.
- **Solo imagen:** una cabeza lineal entrenada solo con z_img (mismos pares).
- **Fusión:** la cabeza multimodal del Bloque 7.

La pregunta que responde: *¿cuánto aporta cada modalidad?*

In [ ]:
# ============================================================
# BLOQUE 8 — ABLACIÓN
# ============================================================
def predecir(modelo, X, bs=256):
    modelo.eval()
    dl = DataLoader(TensorDataset(X), batch_size=bs, shuffle=False)
    ps = []
    with torch.no_grad():
        for (xb,) in dl:
            ps.append(torch.sigmoid(modelo(xb.to(DISPOSITIVO))).cpu())
    return torch.cat(ps).numpy()

def metricas(y, p, umbral=0.5):
    yp = (p >= umbral).astype(int)
    return {'ROC-AUC': roc_auc_score(y, p),
            'Recall': recall_score(y, yp, zero_division=0),
            'Precisión': precision_score(y, yp, zero_division=0),
            'F1': f1_score(y, yp, zero_division=0),
            'Accuracy': accuracy_score(y, yp)}

# --- Solo serie: GRU original ---
p_serie = predecir(modelo_gru, torch.tensor(X_test_b_sc, dtype=torch.float32))

# --- Solo imagen: cabeza lineal sobre z_img (mismos pares, misma validación) ---
modelo_img = nn.Sequential(nn.Dropout(0.3), nn.Linear(N_FEATS_A, 1)).to(DISPOSITIVO)
class Envoltura(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return self.m(x).squeeze(1)
modelo_img_w = Envoltura(modelo_img)
opt_i = torch.optim.Adam(modelo_img.parameters(), lr=1e-3, weight_decay=1e-4)
dl_tr_i = DataLoader(TensorDataset(Z_img_train[idx_tr], y_fus_train[idx_tr]),
                     batch_size=128, shuffle=True)
mejor_li, sin_m = float('inf'), 0
for epoca in range(1, 61):
    modelo_img_w.train()
    for xb, yb in dl_tr_i:
        xb, yb = xb.to(DISPOSITIVO), yb.to(DISPOSITIVO)
        opt_i.zero_grad()
        criterio(modelo_img_w(xb), yb).backward(); opt_i.step()
    modelo_img_w.eval()
    with torch.no_grad():
        lv = criterio(modelo_img_w(Z_img_train[idx_val].to(DISPOSITIVO)),
                      y_fus_train[idx_val].to(DISPOSITIVO)).item()
    if lv < mejor_li:
        mejor_li, sin_m = lv, 0
        torch.save(modelo_img.state_dict(), 'mejor_solo_img.pth')
    else:
        sin_m += 1
    if sin_m >= 10: break
modelo_img.load_state_dict(torch.load('mejor_solo_img.pth', map_location=DISPOSITIVO))
p_img = predecir(modelo_img_w, Z_img_test)

# --- Fusión ---
p_fus = predecir(modelo_fusion, X_fus_test)

y_np = y_fus_test.numpy()
tabla = pd.DataFrame({
    'Solo serie (GRU)':   metricas(y_np, p_serie),
    'Solo imagen (CNN)':  metricas(y_np, p_img),
    'Fusión multimodal':  metricas(y_np, p_fus)}).T.round(4)

print('=' * 70)
print(f'ABLACIÓN — mismo conjunto de prueba ({len(y_np):,} ventanas)')
print('=' * 70)
print(tabla.to_string())
mejor = tabla['ROC-AUC'].idxmax()
print(f'\n🏆 Mejor configuración por ROC-AUC: {mejor}')

---
## BLOQUE 9 — Visualización de la ablación
### ¿Qué hace?
Gráfico comparativo de las tres configuraciones para el informe y el cliente.

In [ ]:
# ============================================================
# BLOQUE 9 — VISUALIZACIÓN
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Curvas de entrenamiento de la fusión
axes[0].plot(hist['train'], label='Entrenamiento', color='#0072B2')
axes[0].plot(hist['val'], label='Validación', color='#E69F00')
axes[0].set_title('La fusión converge sin sobreajuste', fontsize=12, loc='left')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Pérdida (BCE)')
axes[0].legend(); axes[0].spines[['top','right']].set_visible(False)

# Barras de ablación
colores = ['#999999', '#999999', '#0072B2']
vals = tabla['ROC-AUC'].values
barras = axes[1].bar(tabla.index, vals, color=colores)
for b, v in zip(barras, vals):
    axes[1].text(b.get_x() + b.get_width()/2, v + 0.01, f'{v:.3f}',
                 ha='center', fontsize=11, fontweight='bold')
axes[1].set_ylim(0, 1.05)
axes[1].set_title('¿Cuánto aporta cada modalidad? (ROC-AUC, prueba)',
                  fontsize=12, loc='left')
axes[1].spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('ablacion_RamaC.png', dpi=150, bbox_inches='tight')
plt.show()

---
## BLOQUE 10 — Guardar modelo final y conclusiones honestas
### ¿Qué hace?
Guarda los pesos y genera las conclusiones **a partir de las métricas calculadas**
(no de texto fijo). Si la fusión no supera a la mejor rama individual, se reporta
como hallazgo válido, tal como exige el criterio de honestidad de la actividad.

In [ ]:
# ============================================================
# BLOQUE 10 — GUARDAR Y CONCLUIR
# ============================================================
torch.save({'fusion_state_dict': modelo_fusion.state_dict(),
            'dim_img': N_FEATS_A, 'dim_seq': DIM_GRU,
            'metricas_test': tabla.to_dict()}, 'Centinela_Fase2_RamaC_Fusion.pth')

auc_s, auc_i, auc_f = tabla.loc['Solo serie (GRU)','ROC-AUC'], \
                      tabla.loc['Solo imagen (CNN)','ROC-AUC'], \
                      tabla.loc['Fusión multimodal','ROC-AUC']

print('=' * 70)
print('CONCLUSIONES — RAMA C (generadas desde las métricas)')
print('=' * 70)
print(f'Solo serie:  ROC-AUC {auc_s:.4f}')
print(f'Solo imagen: ROC-AUC {auc_i:.4f}')
print(f'Fusión:      ROC-AUC {auc_f:.4f}\n')

if auc_f > max(auc_s, auc_i):
    print('La fusión supera a ambas ramas individuales: las modalidades aportan')
    print('información complementaria bajo la alineación simulada declarada.')
elif auc_f >= max(auc_s, auc_i) - 0.01:
    print('La fusión empata con la mejor rama individual: la modalidad visual,')
    print('bajo alineación simulada, aporta información mayormente redundante')
    print('con la serie. Hallazgo honesto: el valor de la imagen dependerá de')
    print('un co-registro real en campo (propuesto para la Fase 3).')
else:
    print('La fusión NO supera a la mejor rama individual. Hallazgo válido que')
    print('se documenta: bajo alineación simulada, la señal visual añade ruido')
    print('en lugar de información. Un co-registro genuino (bocatoma + fecha)')
    print('es condición necesaria para que la fusión aporte valor real.')

print('\nLimitación declarada: el emparejamiento imagen–ventana es simulado por')
print('condición observable (pH del día t), no un co-registro de campo.')